In [1]:
import pandas as pd
import numpy as np

In [2]:
def load_netflix_file(filepath):

    data = []
    movie_id = None

    with open(filepath, "r") as f:

        for line in f:

            line = line.strip()

            if line.endswith(":"):
                movie_id = int(line[:-1])

            else:

                user_id, rating, date = line.split(",")

                data.append([
                    int(user_id),
                    movie_id,
                    int(rating),
                    date
                ])

    return pd.DataFrame(
        data,
        columns=[
            "user_id",
            "movie_id",
            "rating",
            "date"
        ]
    )

In [3]:
ratings = load_netflix_file(
    "../combined_data_1.txt"
)

In [4]:
sample_ratings = ratings.sample(
    n=1_000_000,
    random_state=42
)

sample_ratings.shape

(1000000, 4)

In [5]:
user_counts = sample_ratings.groupby(
    "user_id"
).size()

active_users = user_counts[
    user_counts >= 20
].index

ratings_filtered = sample_ratings[
    sample_ratings["user_id"].isin(active_users)
].copy()

ratings_filtered.shape

(50954, 4)

In [6]:
ratings_filtered["date"] = pd.to_datetime(
    ratings_filtered["date"]
)

ratings_filtered = ratings_filtered.sort_values(
    ["user_id", "date"]
)

In [7]:
train_list = []
test_list = []

for user_id, group in ratings_filtered.groupby(
    "user_id"
):

    train_list.append(
        group.iloc[:-5]
    )

    test_list.append(
        group.iloc[-5:]
    )

In [8]:
train_df = pd.concat(train_list)

test_df = pd.concat(test_list)

print(train_df.shape)
print(test_df.shape)

(41324, 4)
(9630, 4)


In [9]:
from surprise import Dataset
from surprise import Reader
from surprise import KNNBasic
from surprise import accuracy

In [10]:
reader = Reader(
    rating_scale=(1,5)
)

train_data = Dataset.load_from_df(
    train_df[
        ["user_id","movie_id","rating"]
    ],
    reader
)

trainset = train_data.build_full_trainset()

In [11]:
testset = list(
    zip(
        test_df["user_id"],
        test_df["movie_id"],
        test_df["rating"]
    )
)

In [12]:
sim_options = {
    "name":"cosine",
    "user_based":False
}

item_cf = KNNBasic(
    sim_options=sim_options
)

item_cf.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [13]:
predictions_item = item_cf.test(
    testset
)

In [14]:
item_rmse = accuracy.rmse(
    predictions_item
)

item_rmse

RMSE: 1.1287


np.float64(1.1286635750707035)